# Cleaning text columns

Python stage, lesson P18. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** clean a text column with the `str` accessor and keep the original column; find or extract one pattern with a small regular expression; normalize accented text to NFC before filtering, because two texts that print the same can compare unequal.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import io
import re

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# The menu boards of the three cafes: the 48 lines of menu_accents.csv, version 1 of the companion dataset cafe-menu-accents.
# Each board was typed on a different device, so it is stored in its own way.
MENU_CSV = """menu_item_id,store_id,product_id,board_name
1,1,1,Café espresso
2,1,2,Café americano
3,1,3,Café au lait
4,1,4,Cappuccino
5,1,5,Mocha
6,1,6,Café cold brew
7,1,7,Açaí green tea
8,1,8,Chai latté
9,1,9,Piña colada iced tea
10,1,10,Croissant
11,1,11,Blueberry muffin
12,1,12,Crème brûlée banana bread
13,1,13,Jalapeño bagel with cream cheese
14,1,14,Avocado toast
15,1,15,Turkey sandwich
16,1,16,Pumpkin spice café latte
17,2,1,Café espresso
18,2,2,Café americano
19,2,3,Café au lait
20,2,4,Cappuccino
21,2,5,Mocha
22,2,6,Café cold brew
23,2,7,Açaí green tea
24,2,8,Chai latté
25,2,9,Piña colada iced tea
26,2,10,Croissant
27,2,11,Blueberry muffin
28,2,12,Crème brûlée banana bread
29,2,13,Jalapeño bagel with cream cheese
30,2,14,Avocado toast
31,2,15,Turkey sandwich
32,2,16,Pumpkin spice café latte
33,3,1,Café espresso
34,3,2,Café americano
35,3,3,Café au lait
36,3,4,Cappuccino
37,3,5,Mocha
38,3,6,Café cold brew
39,3,7,Açaí green tea
40,3,8,Chai latté
41,3,9,Piña colada iced tea
42,3,10,Croissant
43,3,11,Blueberry muffin
44,3,12,Crème brûlée banana bread
45,3,13,Jalapeño bagel with cream cheese
46,3,14,Avocado toast
47,3,15,Turkey sandwich
48,3,16,Pumpkin spice café latte
"""
menu = pd.read_csv(io.StringIO(MENU_CSV))

## Unit 1: str: clean a whole column

**Predict first.** the six entries are two drinks, Latte and Cold Brew. How many different texts are in `typed`, and how many are left in `clean`?

In [ ]:
till = pd.DataFrame({"typed": [" Latte", "latte ", "LATTE", "Cold  Brew", "cold brew", "Cold Brew "]})
till = till.assign(clean=till["typed"].str.strip().str.lower())
print(till["typed"].nunique(), till["clean"].nunique())

**Predict first.** after two spaces become one, how many texts are left in `clean`, and does `typed` still hold the six entries as typed?

In [ ]:
till = till.assign(clean=till["clean"].str.replace("  ", " "))
print(till["clean"].nunique())
print(till["typed"].tolist())

## Unit 2: A small regular expression

**Predict first.** the pattern takes the digits in front of `oz`, with or without a space, in any case. What does it return for each note, and for how many notes is there no size?

In [ ]:
notes = pd.Series(["Latte 12oz", "Mocha 16 oz", "Cold Brew 16OZ", "Espresso", "Chai Latte 12 oz", "Latte 16oz"])
sizes = notes.str.extract(r"(\d+)\s*oz", flags=re.IGNORECASE, expand=False)
print(sizes.tolist())
print(sizes.isna().sum())

**Predict first.** four spellings of one drink, with one, two or three spaces and with a space and a tab. How many different texts are left after the replace?

In [ ]:
spellings = pd.Series(["Cold Brew", "Cold  Brew", "Cold   Brew", "Cold \tBrew"])
print(spellings.str.replace(r"\s+", " ", regex=True).nunique())

## Unit 3: NFC: texts that look the same

**Predict first.** the three boards print the name of product 1 the same way. What are the lengths of the three texts?

In [ ]:
name1 = menu[menu["product_id"] == 1]
print(name1["board_name"].tolist())
print(name1["board_name"].str.len().tolist())

**Predict first.** how many lines equal the typed name as stored, and how many after the whole column is normalized to NFC?

In [ ]:
typed_name = "Café espresso"
print((menu["board_name"] == typed_name).sum())
clean_names = menu["board_name"].str.normalize("NFC")
print((clean_names == typed_name).sum())

## Unit 4: Check: one name for each product

**Predict first.** how many different clean names are there, and how many lines does each one have?

In [ ]:
menu = menu.assign(clean_name=menu["board_name"].str.normalize("NFC"))
print(menu["clean_name"].nunique())
per_name = menu.groupby("clean_name").size()
print(per_name.min(), per_name.max())

## Finish it

### Exercise 1: Finish it

**Task:** What is the total size, in ounces, of the five drinks on the notes? Replace the first `None` with the digits in front of `oz` in each note, taken with `str.extract` and a pattern that accepts any number of digits, a space or none, and any case. Then set `answer_1` to the total, with the sizes turned into numbers.

**Expected:** one decimal number.

In [ ]:
notes_2 = pd.Series(["Mocha 16oz", "Latte 12 oz", "Espresso", "Cold Brew 16 OZ", "Mocha 12oz"])
sizes_2 = None  # replace None: the digits before "oz" in each note, with str.extract (digits, a space or none, any case)
answer_1 = None  # then set answer_1 to the total of the sizes in sizes_2, as numbers

In [ ]:
_ = check("Exercise 1", "answer_1", 56.0, traps={16.0: "16 means each size lost its first digit: \\d takes one digit, so 16 became 6 and 12 became 2. Use \\d+ for one or more digits."})

<details>
<summary>Hint</summary>

Unit 2 takes the digits in front of `oz` with `str.extract`. `\d+` is one or more digits, and the extracted digits are text: `astype(float)` makes them numbers.

</details>

<details>
<summary>Solution</summary>

```python
notes_2 = pd.Series(["Mocha 16oz", "Latte 12 oz", "Espresso", "Cold Brew 16 OZ", "Mocha 12oz"])
sizes_2 = notes_2.str.extract(r"(\d+)\s*oz", flags=re.IGNORECASE, expand=False)
answer_1 = sizes_2.astype(float).sum()
```

The answer is 56.0: 16, 12, none, 16 and 12. The missing size of the espresso is skipped by the sum. A pattern with `\d` takes one digit only, so 16 becomes 6 and 12 becomes 2, and the total is 16.0; the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many of the names on Old Town's board also appear on Campus's board? The two boards were typed on different devices. Before you start, say in one sentence how many names you expect, and why. Match the board of store 1 with the board of store 2 on the name, with a promise in `validate`, and count the matches. Write the code in the empty cell below and set `answer_2`.

**Expected:** one whole number.

In [ ]:
_ = check("Exercise 2", "answer_2", 16, traps={6: "6 are the names without an accent. The same name is stored in two ways on the two boards, so the others did not match. Make the names agree before you merge."})

<details>
<summary>Hint</summary>

Unit 3 makes two stored forms of one name agree, and lesson P15 matches two tables on a key with a promise in `validate`. Which column do you match on?

</details>

<details>
<summary>Solution</summary>

```python
board_1 = menu[menu["store_id"] == 1]
board_2 = menu[menu["store_id"] == 2]
names_1 = board_1.assign(name=board_1["board_name"].str.normalize("NFC"))
names_2 = board_2.assign(name=board_2["board_name"].str.normalize("NFC"))
both = names_1.merge(names_2, on="name", validate="one_to_one")
answer_2 = len(both)
```

The answer is 16: every name of Old Town's board is on Campus's board. Merging on `board_name` as stored finds only 6, the names without an accent, because the other 10 are stored in NFC on one board and in NFD on the other; the check cell says so.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants the bare drink names of the Old Town lines that begin with Café, without the word and its space. The list shows a name that is not on the board. Fix the line, then set `answer_3 = bare.tolist()`.

**Expected:** a list of four names.

In [ ]:
cafe_names = menu.loc[(menu["store_id"] == 1) & menu["board_name"].str.startswith("Café"), "board_name"]
bare = cafe_names.str.strip("Café ")  # the colleague's line
answer_3 = None  # fix the line above, then set answer_3 = bare.tolist()

In [ ]:
_ = check("Exercise 3", "answer_3", ["espresso", "americano", "au lait", "cold brew"], traps={("espresso", "mericano", "u lait", "cold brew"): "americano lost its first letter: strip with text removes any of the characters you list, from both ends, not the word. Remove the word, or the prefix, instead."})

<details>
<summary>Hint</summary>

Print `bare`. One name has lost a letter that is not part of the word `Café`. Look up what `str.strip` does with the text you give it, and what `str.removeprefix` and `str.replace` do.

</details>

<details>
<summary>Solution</summary>

```python
cafe_names = menu.loc[(menu["store_id"] == 1) & menu["board_name"].str.startswith("Café"), "board_name"]
bare = cafe_names.str.removeprefix("Café ")
answer_3 = bare.tolist()
```

The answer is `['espresso', 'americano', 'au lait', 'cold brew']`. `strip("Café ")` removes any of the characters C, a, f, é and space from both ends, so it cut the `a` of americano and the `a` of au lait; `removeprefix` removes the word once, and `str.replace("Café ", "", regex=False)` gives the same names. The check cell says so.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")